In [2]:
import numpy as np
def read_cebaf(filename):
    data = np.load(filename)
    nentries = int(data.shape[0]/3)
    heat = data[0:nentries]
    trip = data[nentries:2*nentries]
    alpha = data[2*nentries:3*nentries] 
    return heat,trip,alpha

In [4]:
import os
import matplotlib.pyplot as plt

mf_location = '../drivers/results/index0_agent_MO-KerasTD3-v0_buf_cfg_bsize_cfg_env_PACES-MO-CEBAF-8D-VEC_hasha7f4255_results_20240813-115003'
mb_location = '../drivers/results/index0_agent_MO-KerasMB-v0_buf_cfg_bsize_cfg_env_PACES-MO-CEBAF-8D-VEC-TF_hasha7f4255_results_20240813-113524'

def make_images(mf_location, mb_location, n_epochs=10000):
    os.makedirs("movies", exist_ok=True)
    ga_data = np.load('1L10_TEST8_nsga_II_results.npy')
    ga_index = np.argsort(ga_data[:,0])
    ga_heat = ga_data[:,0]
    ga_trip = ga_data[:,1]
    for i in range(1000, n_epochs+1, 1000):  
        # Join various path components
        mb_file = f'inference_results_steps{i}.npy'
        mf_file = f'inference_results_steps{i+3000}.npy'
        mf_heat, mf_trip, mf_alpha = read_cebaf(os.path.join(mf_location, mf_file))
        mb_heat, mb_trip, mb_alpha = read_cebaf(os.path.join(mb_location, mb_file))
        fig, ax = plt.subplots(dpi=150)
        plt.scatter(mf_heat, mf_trip, c=mf_alpha, marker='o', s=25, edgecolors='black', alpha=0.75, label="Model-Free MORL")
        plt.scatter(mb_heat, mb_trip, c=mb_alpha, marker='*', s=35, edgecolors='red', alpha=0.75, label="Model-Diff MORL")
        plt.plot(ga_heat[ga_index], ga_trip[ga_index], c='black', linestyle='dashed', label="NSGA-II")

        # Format
        plt.legend(loc="upper right")
        plt.grid()
        plt.xlim(20.8,22.2)
        plt.ylim(0.015,0.04)        
        plt.xlabel('Heat Load [W]')
        plt.ylabel('Trip Rate [per hour]');
        plt.colorbar()
        plt.tight_layout()
        plt.title("Step #"+str(i), fontsize=16)
        plt.savefig(f'movies/cebaf_mo'+str(i).zfill(6)+'.png', bbox_inches='tight', dpi=300)
        plt.close()

make_images(mf_location, mb_location, n_epochs=250000)

In [3]:
# import cv2

# def create_video_from_images(folder):
#     video_filename = 'created_video.mp4'
#     valid_images = [i for i in os.listdir(folder) if i.endswith((".jpg", ".jpeg", ".png"))]
#     valid_images = sorted(valid_images)
#     print(valid_images)

#     first_image = cv2.imread(os.path.join(folder, valid_images[0]))
#     h, w, _ = first_image.shape

#     codec = cv2.VideoWriter_fourcc(*'mp4v')
#     vid_writer = cv2.VideoWriter(video_filename, codec, 30, (w, h))

#     for img in valid_images:
#         loaded_img = cv2.imread(os.path.join(folder, img))
#         for _ in range(20):
#             vid_writer.write(loaded_img)

#     vid_writer.release()

ImportError: dlopen(/Users/schram/.local/lib/python3.9/site-packages/cv2/cv2.cpython-39-darwin.so, 0x0002): Library not loaded: /opt/homebrew/opt/ffmpeg/lib/libavcodec.58.dylib
  Referenced from: <41B44FD8-0712-3895-9185-9D9428E8DB4D> /Users/schram/.local/lib/python3.9/site-packages/cv2/cv2.cpython-39-darwin.so
  Reason: tried: '/opt/homebrew/opt/ffmpeg/lib/libavcodec.58.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/ffmpeg/lib/libavcodec.58.dylib' (no such file), '/opt/homebrew/opt/ffmpeg/lib/libavcodec.58.dylib' (no such file), '/usr/local/lib/libavcodec.58.dylib' (no such file), '/usr/lib/libavcodec.58.dylib' (no such file, not in dyld cache)

In [ ]:
# Create video from resized images
create_video_from_images("movies")